# FinTech Pro Activity Drop — Debug Notebook

**Analyst:** Nimrod Fisher | **Date:** 2026-04-28

Standalone debug notebook. Runs top-to-bottom from saved CSVs — no DB connection required.

**Question:** Why did paying FinTech Pro accounts show a ~30% drop in product activity in the most recent 30-day period vs. the prior 30 days?

See [`../plan.md`](../plan.md) for the full analysis plan and checkpoint log.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

ROOT = Path.cwd().parent if Path.cwd().name == 'deliverables' else Path.cwd()
RESULTS = ROOT / 'results'
print('Analysis root:', ROOT)
print('Results folder:', RESULTS)

## Hypotheses & Decision

**Decision supported:** Prioritize CS outreach for at-risk FinTech Pro accounts before renewal.

- **H0 (null):** Drop is a measurement artifact (coverage gap). Refuted if: drop persists on trimmed windows.
- **H1:** Drop driven by a small number of large accounts going dark.
- **H2:** Accounts active at account level but key analyst users stopped logging in.
- **H3:** Drop concentrated in specific event types while others held.
- **H4:** Normal seasonality or platform-wide trend. Refuted if: platform drop < 2× FinTech Pro drop.

**Analysis windows:** Prior 30 = Apr 7–May 6, 2025 | Last 30 = May 7–Jun 6, 2025

## Phase 2 — Data QA

**Overall: PASS WITH CAVEATS**

| Check | Result | Severity |
|-------|--------|----------|
| Event coverage | 2025-03-07 → 2025-06-06, 1,960 rows, 92 days | INFO |
| Plan casing bug | `plan` stores lowercase (`pro` not `Pro`) | MEDIUM (mitigated) |
| FinTech Pro accounts | **2 accounts** (PJohnson Corp, ZGarcia Corp) | MEDIUM — small n |
| Drop confirmed | 37 → 26 events (−29.7%) on matched windows | PASS |
| H0 zero-event tail | Jun 7–17 outside both windows — not the cause | PASS |

In [ ]:
qa_csvs = {p.stem: pd.read_csv(p) for p in sorted((RESULTS / 'qa').glob('*.csv'))}
for name, df in qa_csvs.items():
    print(f'--- {name} ({df.shape[0]} rows x {df.shape[1]} cols)')
    display(df)

## Phase 3 — EDA

**Key findings:**
1. **Weekly trend:** Stable (8-10/wk) → spike May 19 (9) → taper (4, 3). Not a straight-line decline.
2. **Event-type split:** `logout` −60%, `api_call` −63%, `login` −40%. `report_view` +13%, `file_upload` +17%. Session events drove the whole decline.
3. **User level:** ZGarcia all 3 analysts down + both admins dark. PJohnson: 1 analyst down (Olivia −46%), 1 up (Eve +100%).

In [ ]:
df_ts = pd.read_csv(RESULTS / 'eda' / '03_eda-activity-timeseries.csv')
df_ts['period'] = df_ts['week_start'].apply(lambda w: 'Last 30' if w >= '2025-05-07' else 'Prior 30')
colors = ['#6366f1' if p == 'Prior 30' else '#3b82f6' for p in df_ts['period']]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(df_ts)), df_ts['event_count'], color=colors, alpha=0.85)
ax.set_xticks(range(len(df_ts)))
ax.set_xticklabels(df_ts['week_start'], rotation=35, ha='right', fontsize=9)
ax.axvspan(4.5, 8.5, alpha=0.07, color='#3b82f6', label='Last 30 days')
ax.set_title('Weekly Event Count — FinTech Pro')
ax.set_ylabel('Events / week')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_et = pd.read_csv(RESULTS / 'eda' / '04_eda-event-type-breakdown.csv')
display(df_et)
x = range(len(df_et)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - w/2 for i in x], df_et['prior_30'], w, label='Prior 30', color='#6366f1', alpha=0.85)
colors_last = ['#ef4444' if c < 0 else '#22c55e' for c in df_et['pct_change']]
ax.bar([i + w/2 for i in x], df_et['last_30'], w, label='Last 30', color=colors_last, alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(df_et['event_type'])
ax.set_title('Event-Type Breakdown: Prior 30 vs Last 30')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_u = pd.read_csv(RESULTS / 'eda' / '05_eda-account-user-distributions.csv')
display(df_u[['full_name', 'role', 'account_name', 'prior_30_events', 'last_30_events']])

## Phase 4 — Deep Analysis

| Hypothesis | Verdict | Strength |
|-----------|---------|----------|
| H0 — Artifact | **Refuted** | Strong — trimmed: −28.1% |
| H1 — Concentration | **Partial confirm** | ZGarcia = 63.6% of drop |
| H2 — Seat churn | **Partial confirm** | ZGarcia admins 2→0; analysts still present |
| H3 — Event-type shift | **Confirmed** | Two sub-patterns: session thinning vs API shutdown |
| H4 — Seasonality | **Refuted** | Platform +10.8%, 40.5pp divergence |

In [ ]:
df = pd.read_csv(RESULTS / 'deep-analysis' / '06_da-concentration.csv')
print('H1 — Account concentration:')
display(df)

In [ ]:
df = pd.read_csv(RESULTS / 'deep-analysis' / '07_da-seat-churn.csv')
print('H2 — Seat churn by role:')
display(df)

In [ ]:
df_bm = pd.read_csv(RESULTS / 'deep-analysis' / '08_da-platform-benchmark.csv')
display(df_bm)
colors = ['#ef4444' if v < 0 else '#22c55e' for v in df_bm['pct_change']]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_bm['segment'], df_bm['pct_change'], color=colors, alpha=0.85, width=0.5)
ax.axhline(0, color='#94a3b8', linewidth=1.2, linestyle='--')
ax.set_ylabel('% Change'); ax.set_title('H4: FinTech Pro vs All Others')
for i, v in enumerate(df_bm['pct_change']):
    ax.text(i, v + (0.5 if v > 0 else -3), f'{v:+.1f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
df = pd.read_csv(RESULTS / 'deep-analysis' / '09_da-artifact-test.csv')
print('H0 — Artifact test (trimmed windows):')
display(df)

In [ ]:
df = pd.read_csv(RESULTS / 'deep-analysis' / '10_da-event-type-per-account.csv')
print('H3 — Event-type per account:')
display(df)

## Phase 5 — Synthesis

**Overall Conclusion:** The −29.7% drop is real, FinTech-Pro-specific, and caused by two distinct account-level stories.

- **ZGarcia Corp** (64% of drop, −47%): Pre-churn. API stopped ~May 1 (4→0 api_calls). Both admins went dark. Analysts still present but reduced. Zero support tickets. Silent pre-churn pattern.
- **PJohnson Corp** (36% of drop, −18%): Session thinning. Fewer logins/logouts but `report_view` increased. Feature requests filed and resolved. Lower urgency.

**Reframe:** The better question is not 'why did FinTech Pro drop?' but 'what is the right intervention for each account?'

## Phase 6 — Validation

**All headline conclusions survived validation.**

- **Window sensitivity:** Per-day normalized on shifted window: −6%. Headline −29.7% is accurate for the May 7–Jun 6 window; magnitude is partly boundary-dependent.
- **ZGarcia API timeline:** Gradual taper then hard stop after Apr 28. Consistent with deliberate discontinuation, not a technical outage.
- **Support tickets:** PJohnson filed 2 feature_requests (both resolved). ZGarcia filed **zero tickets** — silent disengagement.
- **Composition shift:** No new accounts joined or left the segment during the window.

**Revisions:** ZGarcia urgency elevated. API claim narrowed to 'likely behavioral, requires CS verification'. PJohnson report_view claim narrowed to 'from a low base (2→5)'.

In [ ]:
for csv in sorted((RESULTS / 'validation').glob('*.csv')):
    print(f'--- {csv.stem}')
    display(pd.read_csv(csv))

In [ ]:
df_api = pd.read_csv(RESULTS / 'validation' / 'val-zgarciacorp-api-timeline.csv')
weeks = df_api['week_start'].tolist() + ['May 7+']
counts = df_api['api_calls'].tolist() + [0]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(weeks)), counts, color=['#6366f1']*len(df_api) + ['#ef4444'], alpha=0.85)
ax.set_xticks(range(len(weeks))); ax.set_xticklabels(weeks, rotation=30, ha='right', fontsize=9)
ax.set_title('ZGarcia Corp: api_call Events by Week')
ax.set_ylabel('API calls')
plt.tight_layout(); plt.show()

## Conclusions & Recommendations

### Headline Answer
The −29.7% drop is real, behavioral, and FinTech-Pro-specific. It reflects two accounts with different root causes:

| Account | Change | Risk | Root Cause |
|---------|--------|------|------------|
| ZGarcia Corp | −47% | **Pre-churn** | API stopped, admins dark, no tickets |
| PJohnson Corp | −18% | Monitor | Fewer sessions, core work sustained |

### Recommendations
1. **P1 (Urgent):** CS call to ZGarcia Corp this week. Confirm API status, understand admin turnover, assess renewal intent.
2. **P2 (This month):** Check in with Olivia Garcia (PJohnson's top analyst, −46%). Understand if role change or dissatisfaction before drift accelerates.
3. **P3 (Next quarter):** Grow FinTech Pro segment to ≥5 accounts before drawing segment-level conclusions. n=2 prevents generalization.

## Re-running Against the Database

This notebook uses saved CSVs in `results/`. To re-run any query:
1. Open the matching `.sql` file in `queries/`
2. Execute it via the Supabase MCP (`execute_sql` tool)
3. Overwrite the corresponding CSV in `results/<phase>/`
4. Re-run this notebook to rebuild charts

All SQL filters use `a.industry = 'FinTech' AND a.plan = 'pro'` (lowercase plans).